<a href="https://colab.research.google.com/github/fourmodern/2026_aidrugdiscovery/blob/main/Day06_LLM_Agent/t042_molt5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. 실행 환경 / 요구사항

- **Colab에서 바로 실행됩니다.** GPU(T4)를 켜면 빠르지만, **CPU 런타임에서도 동작**합니다.
- 이 노트북은 `molt5-large` 모델 **2종(caption2smiles, smiles2caption)** 을 내려받습니다. 각각 약 3GB이므로
  첫 실행 시 다운로드에 수 분이 걸립니다. Hugging Face 로그인이나 라이선스 동의는 **필요 없습니다**.
- 생성 시간(참고): CPU 기준 프롬프트 1개당 약 15초, GPU에서는 1~2초.

In [ ]:
# 실행 환경 확인 (Colab: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU)
import torch

print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[안내] CPU 런타임입니다. 실행은 되지만 느립니다.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

# **MolT5**

## Introduction to MolT5

MolT5는 T5 (Text-To-Text Transfer Transformer) 아키텍처를 기반으로 한 분자 언어 모델로, 다양한 화학정보학(Cheminformatics) 작업을 처리하기 위해 설계되었습니다. 이 모델은 분자 설명을 SMILES(분자 구조를 표현하는 문자열 형식), IUPAC 이름, 분자 특성 등의 다양한 표현으로 변환할 수 있습니다. MolT5는 화학 언어와 구조화된 분자 데이터 간의 복잡한 매핑을 학습하는 능력 덕분에 신약 개발과 화학정보학 분야에서 중요한 도구로 활용될 수 있습니다.

MolT5의 원 논문은 Edwards et al., **"Translation between Molecules and Natural Language"** (EMNLP 2022, arXiv:2204.11817) 입니다.
대량의 무표지(unlabeled) SMILES와 텍스트로 T5를 사전학습한 뒤, ChEBI-20 데이터셋으로
**molecule captioning(분자 → 설명)** 과 **text-based de novo molecule generation(설명 → 분자)** 두 과제를 미세조정한 모델입니다.
공개 체크포인트가 `laituan245/molt5-*` 이며, 이 노트북에서는 그 중 `-large` 두 종을 사용합니다.

> ⚠️ 이 노트북의 이전 버전에는 실제로 존재하지 않는 논문 제목이 인용되어 있었습니다.
> 생성형 모델 실습에서는 **모델 출력뿐 아니라 교재의 인용도 검증 대상**이라는 점을 기억하세요.

## MolT5 튜토리얼

### 1. Colab 환경 설정

In [ ]:
# 필요한 라이브러리 설치
# - transformers : T5 계열 모델/토크나이저
# - rdkit        : SMILES 파싱 및 분자 그림
# - sentencepiece: MolT5 토크나이저(T5Tokenizer)가 구버전 transformers에서 요구
!pip install -q transformers rdkit sentencepiece

### 2. 라이브러리 가져오기

In [ ]:
# 필요한 라이브러리 가져오기
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from rdkit.Chem import Draw
from rdkit import Chem
import pandas as pd  # 여러 분자 설명을 처리하기 위한 라이브러리
from IPython.display import display  # Colab에서 이미지를 표시하기 위한 라이브러리

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

### 3. MolT5 모델과 토크나이저 로드하기

In [ ]:
# MolT5 모델과 토크나이저 로드
# molt5-large 는 약 3GB 입니다. 첫 실행 시 다운로드에 수 분 걸릴 수 있습니다.
tokenizer = T5Tokenizer.from_pretrained("laituan245/molt5-large-caption2smiles", model_max_length=512)
model = T5ForConditionalGeneration.from_pretrained("laituan245/molt5-large-caption2smiles").to(device).eval()
print("caption2smiles 모델 로드 완료 (device =", device, ")")

### 4. 하나의 텍스트 설명을 분자로 변환하기

In [ ]:
# 분자에 대한 텍스트 설명 입력 (PDE5 저해제 예시: 실데나필/sildenafil)
input_text = 'The molecule is sildenafil, a selective phosphodiesterase type 5 (PDE5) inhibitor used to treat erectile dysfunction and pulmonary arterial hypertension.'

# 설명을 SMILES로 변환
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
output_ids = model.generate(inputs, max_length=512)
smiles = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("생성된 SMILES:", smiles)

# RDKit 파싱으로 유효성 검사 후 시각화
mol = Chem.MolFromSmiles(smiles)
if mol:
    print("RDKit 파싱 성공 - 유효한 SMILES 입니다. Canonical:", Chem.MolToSmiles(mol))
    display(Draw.MolToImage(mol, size=(300, 300)))
else:
    print("유효한 분자를 생성할 수 없습니다 (RDKit 파싱 실패).")

> **모델의 한계에 대한 안내**
>
> MolT5는 "설명 → SMILES"를 학습한 모델일 뿐, 화합물 데이터베이스를 조회하는 것이 아닙니다.
> 따라서 "sildenafil" 이라고 이름을 주어도 **실제 실데나필 구조가 나오지 않는 경우가 대부분**입니다.
> 이 실습의 목적은 정답 구조를 얻는 것이 아니라, (1) 생성 모델이 화학 문자열을 만들어낸다는 점과
> (2) **생성 결과는 반드시 RDKit 파싱으로 검증해야 한다**는 점을 확인하는 것입니다.
> 실제 구조는 아래 7절에서 정답 SMILES를 직접 넣어 확인합니다.

### 5. 여러 개의 분자 설명을 일괄 처리

In [ ]:
# 여러 개의 분자 설명 예시
descriptions = [
    'A benzene ring substituted with a nitro group at position 1 and a hydroxyl group at position 2.',
    'A molecule consisting of a six-membered ring with two hydroxyl groups at positions 1 and 3.',
    'An aliphatic compound containing a methyl group at the first carbon and an ethyl group at the second carbon.'
]

# 여러 설명을 처리하기 위한 DataFrame 생성
df = pd.DataFrame(descriptions, columns=['Description'])
df['SMILES'] = ''

# 각 설명에 대해 SMILES 생성
for idx, desc in enumerate(df['Description']):
    inputs = tokenizer.encode(desc, return_tensors="pt").to(device)
    output_ids = model.generate(inputs, max_length=512)
    df.at[idx, 'SMILES'] = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 생성된 SMILES 전부에 대해 RDKit 파싱 유효성 검사
df['Valid'] = [Chem.MolFromSmiles(s) is not None for s in df['SMILES']]
df['Canonical'] = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None
                   for s in df['SMILES']]

# DataFrame 표시
display(df)
print(f"유효 SMILES: {int(df['Valid'].sum())} / {len(df)}")

# 유효한 분자만 그리드로 시각화
valid_mols = [Chem.MolFromSmiles(s) for s, v in zip(df['SMILES'], df['Valid']) if v]
if valid_mols:
    display(Draw.MolsToGridImage(valid_mols,
                                 legends=[f"#{i}" for i, v in enumerate(df['Valid']) if v],
                                 molsPerRow=3, subImgSize=(280, 280)))

### 6. 에러처리 및 시각화

In [ ]:
# 분자를 시각화하는 함수 (RDKit 파싱 유효성 검사 포함)
def visualize_molecule(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            print("RDKit 파싱 성공:", Chem.MolToSmiles(mol))
            display(Draw.MolToImage(mol, size=(300, 300)))
        else:
            print(f"유효하지 않은 SMILES: {smiles}")
    except Exception as e:
        print(f"오류: {e}")

# 에러 처리가 포함된 예시 (의도적으로 모호한 설명을 넣습니다)
input_text = 'A molecule with an invalid description.'
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
output_ids = model.generate(inputs, max_length=512)
smiles = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("생성된 SMILES:", smiles)

# 에러 처리를 포함한 시각화
visualize_molecule(smiles)

### 7. SMILES를 설명으로 변환하기

In [ ]:
# SMILES를 설명으로 변환하기 위한 모델 로드 (약 3GB, 첫 실행 시 다운로드)
tokenizer_caption = T5Tokenizer.from_pretrained("laituan245/molt5-large-smiles2caption", model_max_length=512)
model_caption = T5ForConditionalGeneration.from_pretrained("laituan245/molt5-large-smiles2caption").to(device).eval()

# 예시 SMILES 문자열 - 실데나필(비아그라)
smiles_example = "CCCc1nn(C)c2c1nc([nH]c2=O)-c1cc(ccc1OCC)S(=O)(=O)N1CCN(C)CC1"

# 입력 SMILES 자체가 유효한지 먼저 RDKit으로 확인
mol_example = Chem.MolFromSmiles(smiles_example)
print("입력 SMILES 유효성:", mol_example is not None)
display(Draw.MolToImage(mol_example, size=(350, 350)))

inputs = tokenizer_caption.encode(smiles_example, return_tensors="pt").to(device)
output_ids = model_caption.generate(inputs, max_length=512)
description = tokenizer_caption.decode(output_ids[0], skip_special_tokens=True)

print("생성된 설명:", description)
print()
print("[안내] MolT5가 만든 설명이 실제 실데나필 약리작용과 다를 수 있습니다.")
print("       생성형 모델의 설명은 근거 문헌으로 반드시 교차검증해야 합니다.")

### 8. SMILES 문자열 변형

In [ ]:
# 8절에서는 "설명을 바꿔서 유사 구조를 만들어보기"를 해봅니다.
# 먼저 원본 SMILES -> 설명 (smiles2caption)
tokenizer_smiles2caption = tokenizer_caption
model_smiles2caption = model_caption

# 원본 SMILES 예시
original_smiles = "CCCc1nn(C)c2c1nc([nH]c2=O)-c1cc(ccc1OCC)S(=O)(=O)N1CCN(C)CC1"  # 실데나필 (PDE5 저해제)

# SMILES를 설명으로 변환
inputs = tokenizer_smiles2caption.encode(original_smiles, return_tensors="pt").to(device)
output_ids = model_smiles2caption.generate(inputs, max_length=512)
caption = tokenizer_smiles2caption.decode(output_ids[0], skip_special_tokens=True)

print("SMILES에 대한 설명:", caption)

In [ ]:
# 변형된 설명 (예: 실데나필의 피페라진 N-메틸기를 에틸기로 치환하는 유사 구조 변형)
modification = "In this molecule, the N-methyl group on the piperazine ring is replaced by an ethyl group."
combined_caption = caption + " " + modification

print("결합된 설명:", combined_caption)

# Caption-to-SMILES 모델은 3절에서 이미 로드했으므로 재사용합니다
tokenizer_caption2smiles = tokenizer
model_caption2smiles = model

# 변형된 설명을 SMILES로 변환
inputs = tokenizer_caption2smiles.encode(combined_caption, return_tensors="pt").to(device)
output_ids = model_caption2smiles.generate(inputs, max_length=512)
modified_smiles = tokenizer_caption2smiles.decode(output_ids[0], skip_special_tokens=True)

print("변형된 SMILES:", modified_smiles)

# RDKit 파싱 유효성 검사 후 원본과 나란히 시각화
original_mol = Chem.MolFromSmiles(original_smiles)
modified_mol = Chem.MolFromSmiles(modified_smiles)
print("원본 유효:", original_mol is not None, "/ 변형 유효:", modified_mol is not None)

if original_mol and modified_mol:
    img = Draw.MolsToGridImage([original_mol, modified_mol],
                               legends=["Original Molecule", "Modified Molecule"],
                               molsPerRow=2, subImgSize=(320, 320))
    display(img)
else:
    print("원본 또는 변형된 SMILES로부터 유효한 분자를 생성할 수 없습니다.")

## 결론 및 추가 탐색

MolT5는 텍스트와 분자 표현 간의 변환에서 뛰어난 다재다능함을 보여줍니다. 이 모델을 활용하면 화학 반응 예측, 분자 특성 추정 등의 다양한 화학정보학 작업에 적용할 수 있습니다.

### References

* **MolT5 논문**: Edwards, C., Lai, T., Ros, K., Honke, G., Cho, K., & Ji, H. (2022).
  *Translation between Molecules and Natural Language.* EMNLP 2022. arXiv:2204.11817.
  코드/데이터: https://github.com/blender-nlp/MolT5
* **T5 아키텍처**: Raffel, C., Shazeer, N., Roberts, A., et al. (2020).
  *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer.* Journal of Machine Learning Research, 21(140), 1-67.
* **관련 분자 NLP 모델**: Schwaller, P., Laino, T., Gaudin, T., et al. (2019).
  *Molecular Transformer: A Model for Uncertainty-Calibrated Chemical Reaction Prediction.* ACS Central Science, 5(9), 1572-1583.
* **모델 카드**: https://huggingface.co/laituan245/molt5-large-caption2smiles ,
  https://huggingface.co/laituan245/molt5-large-smiles2caption